# Regional composite eddies: S, U and D
Pool **S1+S2 → S, U1+U2 → U, D1+D2 → D** at the eddy-day level **before** reconstruction, centre averaging, and whole-eddy bootstrap uncertainty. `Subregion` retains the original six labels. This does not average two precomputed regional means or give unequal-sized subregions equal weight.

The main visual comparisons are four **2×3 figures**: shallow on the top row, deep on the bottom, and S/U/D across columns. First show pooled-Rossby tilt; then overlay low/high |Ro| with **low solid and high dashed**, keeping AE red and CE blue. Repeat both for tilt divided by each day's surface Rc.

There are 6 whole-region AE/CE baselines, 12 shallow/deep AE/CE groups with all Rossby numbers pooled, and 24 shallow/deep/low-high/AE-CE groups. Days with unknown Ro enter the pooled groups but not low/high groups. Every eddy-day is reconstructed once and accumulated into its relevant groups.

Shallow/deep uses deepest valid fitted depth ≤/>1,000 m; low/high uses |surface Ro| < / ≥0.5. Group labels are day-specific, and tracks can cross regions/classes. Existing tilt measurements, PV dominance, shelf class and stratification do not filter the population. Tilt is surface-to-depth displacement of the mean constituent centre in geographic east/north. Velocities remain in native model-grid axes; geographic sections rotate both sampling points and vectors.

Uncertainty resamples whole eddies while retaining equal-day weighting. Variance describes member spread; confidence intervals describe uncertainty in the mean, excluding ESP fit error. Sparse depths are flagged without discarding otherwise valid days. Exact fitted depths are used with no vertical interpolation.


In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
HERE=Path.cwd().resolve()
ANALYSIS=next((p for p in (HERE,*HERE.parents) if (p/'seacofs_tilt_tools.py').exists()),None)
if ANALYSIS is None: raise FileNotFoundError('Run from seacofs_eddy_tilt_analysis or a subfolder')
WORK=ANALYSIS/'esp_population_composites'
DATASET_SRC=ANALYSIS.parent/'seacofs_eddy_dataset_modular'/'src'
ESP_ROOT=Path('/home/z5297792/ESP_zonodo')
for p in (ANALYSIS,WORK,DATASET_SRC,ESP_ROOT):
    if str(p) not in sys.path: sys.path.insert(0,str(p))
import functions as esp
import seacofs_tilt_tools as tilt
import planetary_composite_tools as pct
import composite_comparison_tools as ccomp
import regional_composite_tools as rct
pd.set_option('display.max_columns',60)


In [ ]:
REGION_GROUPS=list(rct.REGION_GROUPS)  # S, U, D
# Population/analysis controls: none of the diagnostics below adds a PV/shelf/N2 filter.
SPLIT_DEPTH_M=1000.
ROSSBY_SPLIT=.5
DOMINANCE_FACTOR=2.
SHELF_LON=154.75
ELLIPSE_FRAC=1.
USE_PV_CACHE=True
BOOTSTRAPS=500
SEED=731
MIN_PLOT_EDDIES=2
SPARSE_EDDIES=20  # visual warning only, not an input filter
WIDTH_KM,RES_KM=400.,10.
RUN_VELOCITY_COMPOSITES=True
RUN_COMPOSITE_FITS=True
SUMMARY_TARGET_DEPTH_M=850.
DEPTH_TOLERANCE_M=100.
MATCH_SHALLOW_TARGETS_M=[200.,500.,850.]
MATCH_DEEP_TARGETS_M=[200.,500.,850.,1500.]
# Load existing corrected N2 diagnostics if available; never compute a new N2 cache here.
LOAD_STRATIFICATION_DIAGNOSTICS=True
N2_CACHE=Path('/srv/scratch/z5297792/SEACOFS_26yr_eddy_dataset/tilt_mechanisms/n2_eddy_day_v4_potential_density_core.parquet')
MIN_N2_CORE_VALID_FRACTION=.80


In [ ]:
paths=tilt.Paths()
grid=tilt.load_grid(paths.grid,paths.z_r)
surface,_=tilt.load_tilt_tables(paths,add_regions=True,grid=grid)
surface=tilt.add_pv_gradient_terms(surface,grid,core_mean=True,frac=ELLIPSE_FRAC,
    surface_method='esp_gaussian',averaging='nonlinear',use_cache=USE_PV_CACHE)
# Refresh the established six labels even if PV was loaded from its saved table.
surface=tilt.add_region_labels(surface,grid)
surface=rct.combine_region_labels(surface)
vertical=tilt.load_vert(paths)
if LOAD_STRATIFICATION_DIAGNOSTICS and N2_CACHE.exists():
    sys.path.insert(0,str(ANALYSIS/'tilt_mechanisms'))
    import mechanism_tools as mech
    n2=mech.load_stratification_cache(N2_CACHE)
    fields=['Eddy','Day']+[c for c in n2 if c in [
        'N2_200m_core_s2','N2_500m_core_s2','N2_200m_core_valid_fraction','N2_500m_core_valid_fraction']]
    surface=surface.merge(n2[fields],on=['Eddy','Day'],how='left',validate='one_to_one')
    for z in [200,500]:
        coverage=f'N2_{z}m_core_valid_fraction'
        surface[f'N2_{z}m_s2']=surface[f'N2_{z}m_core_s2'].where(
            surface[coverage].ge(MIN_N2_CORE_VALID_FRACTION)) if coverage in surface else np.nan
    print('Loaded existing N2 cache; insufficient/unknown coverage stays unavailable.')
else:
    print('Stratification diagnostics unavailable or disabled; all other analysis continues.')
selection_audit,profile_audit,centred_profiles=rct.prepare_population(surface,vertical,
    split_depth=SPLIT_DEPTH_M,ro_split=ROSSBY_SPLIT,dominance_factor=DOMINANCE_FACTOR,shelf_lon=SHELF_LON,regions=REGION_GROUPS)
centred_profiles=rct.bottom_audit(centred_profiles,grid)
display(selection_audit.groupby('selection_status').size().rename('eddy_days'))
display(profile_audit.groupby(['Region','Cyc','profile_status']).size().rename('eddy_days'))


## Population inventory before reconstruction
Counts below include all 42 slots, including zero-count groups. Unknown Rossby numbers are reported in baselines. The same eddy may appear in several rows, so unique-eddy counts must not be summed across groups.

Local bottom checks use the original fitted centre at every depth and the nearest wet model-cell bathymetry. They flag below-bottom fitted levels and unavailable/out-of-domain checks; they **do not filter** the profiles or apply a bathymetric mask to the idealised ESP velocities. Bottom depth at the centre alone does not guarantee support across the entire eddy footprint.


In [ ]:
population_inventory=rct.inventory(profile_audit,regions=REGION_GROUPS,pool_rossby=True)
with pd.option_context('display.max_rows',None): display(population_inventory)
count_matrix=population_inventory.loc[population_inventory.cohort.ne('all') & population_inventory.Ro_class.ne('all')].pivot(
    index='Region',columns=['cohort','Ro_class','Cyc'],values='eddies').reindex(REGION_GROUPS)
fig,ax=plt.subplots(figsize=(12,4),constrained_layout=True)
im=ax.imshow(count_matrix.to_numpy(),aspect='auto',cmap='viridis')
ax.set(yticks=np.arange(len(count_matrix)),yticklabels=count_matrix.index,
       xticks=np.arange(len(count_matrix.columns)),xticklabels=[' / '.join(c) for c in count_matrix.columns])
plt.setp(ax.get_xticklabels(),rotation=35,ha='right')
for i in range(len(count_matrix)):
    for j in range(len(count_matrix.columns)):ax.text(j,i,int(count_matrix.iloc[i,j]),ha='center',va='center',color='white')
fig.colorbar(im,ax=ax,label='Distinct eddies');ax.set_title('Candidate detailed groups before reconstruction')
plt.show()
bottom_check=centred_profiles.groupby('Depth').agg(
    fitted_levels=('Day','size'),checked_levels=('below_local_bottom','count'),
    below_bottom_fraction=('below_local_bottom','mean'))
display(bottom_check)


## Construct each unique eddy-day once
All fields use paired finite u/v values and per-cell denominators. A contributing centre is retained only if the day supplied finite velocity somewhere at that level. With velocity reconstruction disabled, the mode is explicitly centre-only and no composite ESP fit or velocity plot is produced. Profile validity is unchanged from the breakdown notebook.

`regional_results[(region, cohort, Ro_class, Cyc)]` is the main container. Baselines use `('all','all')` for cohort and Ro class; pooled depth groups use `('shallow','all')` or `('deep','all')`. For example: `regional_results[('U','deep','high','CE')]`.


In [ ]:
a=np.arange(-WIDTH_KM/2,WIDTH_KM/2+RES_KM/2,RES_KM)
X,Y=np.meshgrid(a,a)
regional_results,reconstruction_audit=rct.build_composites(profile_audit,centred_profiles,X,Y,esp,
    grid_angle=grid.angle,velocity=RUN_VELOCITY_COMPOSITES,pool_rossby=True)
if not regional_results:raise ValueError('No contributing regional profiles')
rct.add_statistics(regional_results,n_boot=BOOTSTRAPS,seed=SEED)
centre_stats=rct.table(regional_results)
normalised_centre_stats=rct.table(regional_results,'normalised_stats')
contributor_inventory=pd.DataFrame([
    dict(zip(rct.KEYS,key),eddy_days=len(r['members'][['Eddy','Day']].drop_duplicates()),
         eddies=r['members'].Eddy.nunique(),mode=r['mode']) for key,r in regional_results.items()])
display(reconstruction_audit.groupby('reconstruction_status').size().rename('eddy_days'))
display(contributor_inventory)
print('Groups with contributors:',len(regional_results),'of 42')


## Physical displacement: shallow/deep × S/U/D
The first figure pools all Rossby numbers; the second splits low/high using solid/dashed lines. Both plot the norm of the mean constituent displacement in km with pointwise 95% whole-eddy bootstrap intervals. Colours identify AE/CE. Open circles flag fewer than `SPARSE_EDDIES` contributors.

Rows share depth limits within a cohort, and all six panels use a common distance scale within each figure. Group-specific support, mean individual distances and coherence remain in `centre_stats`.


In [ ]:
for split_ro in [False,True]:
    fig,axs=rct.plot_combined_regions(regional_results,split_rossby=split_ro,
        min_eddies=MIN_PLOT_EDDIES,sparse_eddies=SPARSE_EDDIES)
    plt.show()
display(centre_stats[list(rct.KEYS)+['Depth','n_eddy_days','n_eddies','coherence',
                                   'bottom_checked_days','below_bottom_fraction']])


In [ ]:
# Change these controls to inspect one region's low/high-Ro differences in detail.
FOCUS_REGION='D'
FOCUS_COHORT='deep'
FOCUS_RO_CLASS='high'
SHOW_FOCUS_DIAGNOSTICS=False
if SHOW_FOCUS_DIAGNOSTICS:
    rct.plot_group_vectors(regional_results,FOCUS_REGION,FOCUS_COHORT,min_eddies=MIN_PLOT_EDDIES)
    plt.show()
    # fig,axs=plt.subplots(1,2,figsize=(10,4),constrained_layout=True)
    # for ro,style in [('low','-'),('high','--')]:
    #     for cyc in ['AE','CE']:
    #         r=regional_results.get((FOCUS_REGION,FOCUS_COHORT,ro,cyc))
    #         if r is None:continue
    #         s=r['stats']
    #         axs[0].plot(s.n_eddies,s.Depth,style,color=rct.COLORS[cyc],label=f'{cyc} {ro}')
    #         axs[1].plot(s.below_bottom_fraction,s.Depth,style,color=rct.COLORS[cyc],label=f'{cyc} {ro}')
    # for ax,label in zip(axs,['Distinct contributing eddies','Fraction of checked centres below local bottom']):
    #     ax.set(xlabel=label,ylabel='Depth (m)');
    #     if ax.lines:ax.legend(); ax.invert_yaxis()
    # plt.show()


## Tilt normalised by each day's surface Rc
Divide each member's east/north displacement by **its own fitted surface Rc before averaging**. Then take the norm of that averaged dimensionless vector and bootstrap over entire eddies. This is not the kilometre composite displacement divided by a mean radius.

Only the centre displacements are normalised. The velocity composites and their full ESP fits remain in physical kilometres and m/s. Each valid surface profile already requires a finite positive Rc. No new size-based selection is introduced.

The following two figures repeat the same S/U/D shallow/deep layout, with and without the Rossby split.


In [ ]:
for split_ro in [False,True]:
    fig,axs=rct.plot_combined_regions(regional_results,split_rossby=split_ro,normalised=True,
        min_eddies=MIN_PLOT_EDDIES,sparse_eddies=SPARSE_EDDIES)
    plt.show()


## Summary at one common physical depth
Resolve the requested depth once against the dataset's fitted depth grid. Every group is then assessed at that exact level. A group missing that level stays unavailable; it is never assigned its own nearest depth. Available but sparse rows are labelled. This avoids letting one sparse regional group choose a different reference depth for everyone.


In [ ]:
targets=sorted(set([SUMMARY_TARGET_DEPTH_M]+MATCH_SHALLOW_TARGETS_M+MATCH_DEEP_TARGETS_M))
depth_map=rct.resolve_depths(centred_profiles.Depth.unique(),targets,DEPTH_TOLERANCE_M)
display(depth_map)
summary_depth=float(depth_map.set_index('target_m').loc[SUMMARY_TARGET_DEPTH_M,'Depth'])
tilt_summary=rct.summary_at_depth(regional_results,summary_depth,min_eddies=MIN_PLOT_EDDIES)
with pd.option_context('display.max_rows',None):display(tilt_summary)


## Fixed-membership sensitivity
Within each detailed group, keep only days present at **every resolved target depth plus the surface**. Shallow and deep groups use separate target lists because shallow profiles cannot reach the deeper targets. Across regions/Rossby/polarity groups, the target list is identical for the same depth cohort.

The matched curves are evaluated only at those target levels, so membership is truly fixed along each plotted curve. The normalised estimator is recomputed on the same matched days. Empty matches and unresolved targets are reported without relaxing the rule. This sensitivity analysis is intentionally selective; the original available-day composites remain the main results.


In [ ]:
matched_results,matched_audit=rct.fixed_membership(regional_results,depth_map,
    MATCH_SHALLOW_TARGETS_M,MATCH_DEEP_TARGETS_M,n_boot=BOOTSTRAPS,seed=SEED)
display(matched_audit)
matched_centre_stats=rct.table(matched_results)
matched_normalised_stats=rct.table(matched_results,'normalised_stats')
fig,axs=plt.subplots(1,2,figsize=(11,5),constrained_layout=True)
for ro in ['low','high']:
    for cyc in ['AE','CE']:
        key=(FOCUS_REGION,FOCUS_COHORT,ro,cyc)
        if key not in matched_results:continue
        m=matched_results[key]
        for ax,field,col in [(axs[0],'stats','distance_km'),(axs[1],'normalised_stats','distance_Rc')]:
            s=m[field];orig=regional_results[key][field]
            orig=orig.loc[orig.Depth.isin(s.Depth)]
            style='-' if ro=='low' else '--'
            ax.plot(orig[col],orig.Depth,style,color=rct.COLORS[cyc],alpha=.3,label=f'{cyc} {ro}: available')
            ax.plot(s[col],s.Depth,style+'o',color=rct.COLORS[cyc],label=f'{cyc} {ro}: fixed')
            lo,hi=('distance_ci_low','distance_ci_high') if col=='distance_km' else ('distance_Rc_ci_low','distance_Rc_ci_high')
            ax.fill_betweenx(s.Depth,s[lo],s[hi],color=rct.COLORS[cyc],alpha=.12)
for ax,label in zip(axs,['Mean-vector tilt (km)','Mean-vector tilt / surface Rc']):
    ax.set(xlabel=label,ylabel='Depth (m)');ax.invert_yaxis()
    if ax.lines:ax.legend(fontsize=7)
fig.suptitle(f'{FOCUS_REGION}, {FOCUS_COHORT}: available-day versus fixed-membership profiles')
plt.show()


## Environmental context, without additional splits
PV regime, shelf longitude class, radius, water depth and available 0–200/0–500 m stratification summaries use the actual contributing days. PV fractions use the same dominance factor as before, **without** the extra shallow/deep-water regime filters from the planetary/topographic notebooks. N2 is descriptive and coverage-qualified; missing N2 does not remove days. These population summaries help identify possible confounding and are not controlled causal comparisons.


In [ ]:
population_context=rct.population_context(profile_audit,regional_results)
with pd.option_context('display.max_rows',None):display(population_context)


## Full ESP fits to the regional composites
Fit each detailed composite velocity field, using the existing DOPPIO inner + ESP outer procedure. `w`, `Omega` and `Rc` describe the **composite fit**, not averages of member parameters. The plots retain the current absolute-|w|/|Omega| style. Signed parameters, full-fit central vorticity and fit quality remain in the table.

Fits are attempted for all nonempty detailed groups. Failures/unsupported levels leave gaps. Parameter curves do not have confidence bands: the centre bootstrap does not quantify ESP-refit uncertainty. `FIT_REGIONS` can limit fitting for an exploratory run, but defaults to all three. `FIT_PLOT_REGIONS` controls only which figures are displayed.


In [ ]:
FIT_REGIONS=REGION_GROUPS
FIT_PLOT_REGIONS=[FOCUS_REGION]  # set to REGION_GROUPS to show all six populations of populations
comparison_sets=rct.comparison_sets(regional_results,FIT_REGIONS)
composite_esp_fits=pd.DataFrame();composite_fit_audit=pd.DataFrame()
if RUN_COMPOSITE_FITS:
    composite_esp_fits,composite_fit_audit=ccomp.fit_collections(comparison_sets,X,Y,esp,
        min_eddies=MIN_PLOT_EDDIES,transect_radius_km=30.,rho_min_km=30.,rho_max_km=200.,
        out_core_fac=1.75,max_jump_km=100.,min_cell_days=1)
    display(composite_fit_audit)
    if not composite_esp_fits.empty:
        display(composite_esp_fits)
        display(composite_esp_fits.loc[~composite_esp_fits.fit_ok,['population','cohort','Cyc','Depth','reason']])
        for region in FIT_PLOT_REGIONS:
            for ro in ['low','high']:
                label=f'{region} | {ro} Ro'
                if composite_esp_fits.population.eq(label).any():
                    ccomp.plot_fit_profiles(composite_esp_fits,label);plt.show()
else:print('Full composite ESP fits skipped')


## Horizontal and geographic vertical sections
Use the focus region/cohort/Rossby controls above. The horizontal maps use native model-grid axes and share an AE/CE speed scale. The two vertical cuts are geographic **zonal (meridional velocity)** and **meridional (zonal velocity)** through the surface centre. Sampling locations and velocities are rotated together; interpolation is horizontal only. Cuts do not follow a deeper centre that moves away from the surface reference.

The controls select visualisations only: all nonempty populations have already been reconstructed. Missing depths/groups remain labelled, not replaced with a neighbouring population or depth.


In [ ]:
MAP_TARGET_DEPTH_M=500.
map_depth_map=rct.resolve_depths(centred_profiles.Depth.unique(),[MAP_TARGET_DEPTH_M],DEPTH_TOLERANCE_M)
display(map_depth_map)
map_depth=float(map_depth_map.Depth.iloc[0])
fig=rct.horizontal_maps(regional_results,X,Y,FOCUS_REGION,FOCUS_COHORT,FOCUS_RO_CLASS,map_depth)
if fig is not None:plt.show()
else:print('No horizontal velocity map for this exact depth/group; check support and reconstruction mode.')
section_results={(cohort,cyc):r for (region,cohort,ro,cyc),r in regional_results.items()
    if region==FOCUS_REGION and ro==FOCUS_RO_CLASS}
fig=ccomp.plot_sections(section_results,X,Y,f'{FOCUS_REGION} | {FOCUS_RO_CLASS} Ro',FOCUS_COHORT,
    rotation_rad=grid.angle,frame='geographic',min_eddies=MIN_PLOT_EDDIES)
if fig is not None:plt.show()
else:print('No reconstructed velocity sections for this group.')


## Optional output tables
The 42-slot inventory, input/reconstruction audits, depth support, physical and normalised statistics, matched checks, environmental context and full ESP fits remain available as named dataframes. `regional_results` retains member centres and velocity arrays. Exports below overwrite only the regional tables/settings in their own directory; no pickle cache or raw velocity volume is written.


In [ ]:
SAVE_TABLES=False
if SAVE_TABLES:
    out=Path('/srv/scratch/z5297792/SEACOFS_26yr_eddy_dataset_modular/regional_composite_tilt')
    out.mkdir(parents=True,exist_ok=True)
    for name in ['population_inventory','contributor_inventory','reconstruction_audit','centre_stats',
                 'normalised_centre_stats','tilt_summary','depth_map','matched_audit','matched_centre_stats',
                 'matched_normalised_stats','population_context','composite_esp_fits','composite_fit_audit']:
        globals()[name].to_csv(out/f'{name}.csv',index=False)
    (out/'settings.json').write_text(json.dumps(dict(regions=REGION_GROUPS,region_group_map=rct.REGION_GROUP_MAP,split_depth_m=SPLIT_DEPTH_M,
        rossby_split=ROSSBY_SPLIT,rossby='absolute surface w/f; high includes equality',weighting='equal day',
        dominance_factor=DOMINANCE_FACTOR,shelf_lon=SHELF_LON,grid_angle_rad=float(grid.angle),
        bootstrap_draws=BOOTSTRAPS,seed=SEED,minimum_plot_eddies=MIN_PLOT_EDDIES,sparse_eddies=SPARSE_EDDIES,
        summary_target_m=SUMMARY_TARGET_DEPTH_M,depth_tolerance_m=DEPTH_TOLERANCE_M,
        match_shallow_targets_m=MATCH_SHALLOW_TARGETS_M,match_deep_targets_m=MATCH_DEEP_TARGETS_M,
        width_km=WIDTH_KM,resolution_km=RES_KM,velocity=RUN_VELOCITY_COMPOSITES,
        fit_regions=FIT_REGIONS,fit_enabled=RUN_COMPOSITE_FITS,n2_cache=str(N2_CACHE),
        n2_min_valid_fraction=MIN_N2_CORE_VALID_FRACTION),indent=2))
    print(out)
